In [2]:
import json
from pathlib import Path
from datasets import Dataset, Features, Value
from typing import List # Added for the load_txt_lines function

# Dynamically find the monorepo root (where pyproject.toml is)
_p = Path.cwd().resolve()
DATA_DIR = None
while _p != _p.parent:
    if (_p / "pyproject.toml").is_file():
        DATA_DIR = _p / "data"
        break
    _p = _p.parent
if DATA_DIR is None:
    DATA_DIR = Path.cwd().resolve() / "data"


def load_txt_lines(data_dir: Path) -> List[str]:
    lines = []
    for path in sorted(data_dir.glob("*.txt")):
        try:
            raw = path.read_text(encoding="utf-8")
            for line in raw.splitlines():
                s = line.strip()
                if s:
                    lines.append(s)
        except OSError as e:
            print(f"  Warning: could not read {path}: {e}")
    return lines

# Load the raw text data
raw_data = load_txt_lines(DATA_DIR)

print(f"Loaded {len(raw_data):,} lines from .txt files in {DATA_DIR}")

Loaded 663,678 lines from .txt files in /Users/roqqu/Desktop/Build An LLM/build-an-llm/data


In [4]:
# Create a Hugging Face Dataset
# For a simple text dataset, we define a single 'text' feature.
features = Features({
    'text': Value(dtype='string', id=None)
})

hf_dataset = Dataset.from_dict({"text": raw_data}, features=features)

print(f"Created Hugging Face Dataset with {len(hf_dataset)} examples.")
print("Sample example:", hf_dataset[0])

Created Hugging Face Dataset with 663678 examples.
Sample example: {'text': 'WORLD FANTASY, AWARD-WINNING AUTHOR INNEDI 3% OKORAFOR KATA “There’s more vivid imagination in a-page of Nnedi “Okorafor’s work than in whole volumes of" ordinary fantasy epics.” —UrsuLa K.'}


In [6]:
# Install Hugging Face Hub library if not already installed
try:
    import huggingface_hub
except ImportError:
    print("Installing huggingface_hub...")
    !pip3 install huggingface_hub --quiet
    import huggingface_hub
    print("huggingface_hub installed.")

# You will need to log in to Hugging Face Hub
# You can do this by running `huggingface-cli login` in your terminal,
# or by using notebook_login() here:
# from huggingface_hub import notebook_login
# notebook_login()

In [7]:
## Upload to Hugging Face Hub

# IMPORTANT: Replace "your-username/your-dataset-name" with your desired repository ID!
repository_id = "theKingslee/9ja-bookcorpus"  # <--- REPLACE THIS

try:
    hf_dataset.push_to_hub(repository_id)
    print(f"Dataset pushed to https://huggingface.co/datasets/{repository_id}")
except Exception as e:
    print(f"Error pushing to hub: {e}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  4.58ba/s]
Processing Files (1 / 1): 100%|██████████| 33.1MB / 33.1MB, 11.4kB/s  
New Data Upload: 100%|██████████| 33.1MB / 33.1MB, 11.4kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:32<00:00, 32.55s/ shards]


Dataset pushed to https://huggingface.co/datasets/theKingslee/9ja-bookcorpus
